# AI Agent Workflow - Municipal Civic Assistant

**Team: Group1**

**Team members:** 
- Ce Chen, 9007166
- Zhuoran Zhang, 9048508
- Haibo Yuan, 9010929

**GitHub Link:** https://github.com/chence/AI_Agent_Workshop.git

## Workflow Overview

In this notebook, the workflow has four main steps:

1. Load and clean the local public-service dataset.
2. Retrieve the most relevant service entries for a user question.
3. Use simple tool functions to decide ownership and next steps.
4. Evaluate the results on a small benchmark set and save the outputs.

In [1]:
from pathlib import Path
import json
import os
import re

import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("day2_workshop")

DATA_DIR = ROOT / "data"
EVAL_DIR = ROOT / "eval"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

if load_dotenv is not None:
    load_dotenv(ROOT / ".env")

catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
eval_df = pd.read_csv(EVAL_DIR / "service_eval_set.csv")

print("Root:", ROOT.resolve())
print("Catalog rows:", len(catalog))
print("Eval rows:", len(eval_df))

Root: /Users/chrischen/work/CSCN8010/AI_Agent_Workshop
Catalog rows: 15
Eval rows: 8


## Output Schema And Hyperparameters

For this task, the answer should be structured and easy to evaluate. We also keep a few simple hyperparameters so that the workflow is easier to adjust and compare later.

In [2]:
RESPONSE_SCHEMA = {
    "service_name": "string",
    "jurisdiction_level": "City | Region | Province | Federal | Mixed | Unclear",
    "responsible_body": "string",
    "confidence": "float in [0, 1]",
    "reasoning_summary": "short grounded explanation",
    "next_steps": ["step 1", "step 2"],
    "sources": ["url 1", "url 2"],
}

HYPERPARAMS = {
    "top_k": 3,
    "min_score": 1,
    "ambiguity_margin": 1,
    "default_confidence": 0.78,
    "unclear_confidence": 0.35,
}

print(json.dumps(RESPONSE_SCHEMA, indent=2))
print(json.dumps(HYPERPARAMS, indent=2))

{
  "service_name": "string",
  "jurisdiction_level": "City | Region | Province | Federal | Mixed | Unclear",
  "responsible_body": "string",
  "confidence": "float in [0, 1]",
  "reasoning_summary": "short grounded explanation",
  "next_steps": [
    "step 1",
    "step 2"
  ],
  "sources": [
    "url 1",
    "url 2"
  ]
}
{
  "top_k": 3,
  "min_score": 1,
  "ambiguity_margin": 1,
  "default_confidence": 0.78,
  "unclear_confidence": 0.35
}


## Data Preparation And Retrieval

The local service catalog is used as the main knowledge source in this notebook. We normalize the text fields first so the retrieval step can match user questions more reliably.

In [3]:
def normalize_catalog(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["service_name_normalized"] = out["service_name"].str.lower().str.strip()
    out["keywords_list"] = out["keywords"].fillna("").apply(
        lambda value: [item.strip().lower() for item in value.split(";") if item.strip()]
    )
    out["retrieval_text"] = (
        out["service_name"].fillna("") + " | " +
        out["description"].fillna("") + " | " +
        out["keywords"].fillna("") + " | " +
        out["responsible_body"].fillna("")
    ).str.lower()
    return out

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_retrieve(query: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    q_tokens = set(tokenize(query))
    scored_rows = []
    for _, row in df.iterrows():
        row_tokens = tokenize(str(row.get("retrieval_text", "")))
        score = sum(token in row_tokens for token in q_tokens)
        scored_rows.append({**row.to_dict(), "retrieval_score": int(score)})
    ranked = pd.DataFrame(scored_rows).sort_values(
        by=["retrieval_score", "service_name"],
        ascending=[False, True],
    )
    ranked = ranked[ranked["retrieval_score"] >= HYPERPARAMS["min_score"]]
    return ranked.head(top_k).reset_index(drop=True)

clean_catalog = normalize_catalog(catalog)
clean_catalog.head(3)

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url,service_name_normalized,keywords_list,retrieval_text
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/,garbage pickup,"[garbage, waste, pickup, missed collection, cu...",garbage pickup | residential garbage collectio...
1,recycling collection,Region,Region of Waterloo Waste Management,Blue box recycling collection and schedule inf...,recycling;blue box;collection,Check the regional collection calendar and rec...,https://www.regionofwaterloo.ca/,recycling collection,"[recycling, blue box, collection]",recycling collection | blue box recycling coll...
2,property tax billing,City,City of Kitchener Revenue Division,"Property tax billing, payment options, and acc...",property tax;tax bill;billing;assessment,Visit the city's property tax page or contact ...,https://www.kitchener.ca/,property tax billing,"[property tax, tax bill, billing, assessment]","property tax billing | property tax billing, p..."


## Tool Functions And Local Agent Workflow

Instead of asking the model to answer everything directly, we use a few simple tool functions. The workflow first finds candidate services, then checks the likely owner, and finally returns suggested next steps.

In [4]:
def search_service_index(query: str) -> list[dict]:
    results = keyword_retrieve(query, clean_catalog, top_k=HYPERPARAMS["top_k"])
    columns = [
        "service_name",
        "jurisdiction_level",
        "responsible_body",
        "description",
        "next_steps_hint",
        "source_url",
        "retrieval_score",
    ]
    return results[columns].to_dict(orient="records") if len(results) else []

def lookup_service_owner(service_name: str) -> dict:
    matches = clean_catalog[
        clean_catalog["service_name"].str.lower() == service_name.lower()
    ]
    if len(matches) == 0:
        return {
            "service_name": service_name,
            "jurisdiction_level": "Unclear",
            "responsible_body": "Unknown",
            "reasoning_summary": "No exact local service match was found in the catalog.",
            "sources": [],
        }
    row = matches.iloc[0]
    return {
        "service_name": row["service_name"],
        "jurisdiction_level": row["jurisdiction_level"],
        "responsible_body": row["responsible_body"],
        "reasoning_summary": row["description"],
        "sources": [row["source_url"]],
    }

def suggest_next_steps(service_name: str) -> dict:
    matches = clean_catalog[
        clean_catalog["service_name"].str.lower() == service_name.lower()
    ]
    if len(matches) == 0:
        return {
            "next_steps": [
                "Search the official government service page before contacting an office.",
                "If the issue is urgent, call the responsible service centre directly.",
            ]
        }
    row = matches.iloc[0]
    return {
        "next_steps": [
            row["next_steps_hint"],
            f"Verify the latest details on the official source: {row['source_url']}",
        ]
    }

def route_question(question: str) -> dict:
    retrieved = keyword_retrieve(question, clean_catalog, top_k=HYPERPARAMS["top_k"])
    if retrieved.empty:
        return {
            "service_name": "Unclear",
            "jurisdiction_level": "Unclear",
            "responsible_body": "Unknown",
            "confidence": HYPERPARAMS["unclear_confidence"],
            "reasoning_summary": "The local catalog did not contain a strong match for this question.",
            "next_steps": [
                "Ask a more specific question with the service name or location.",
                "Check the relevant municipal, provincial, or federal website directly.",
            ],
            "sources": [],
            "retrieved_candidates": [],
        }

    best = retrieved.iloc[0]
    owner = lookup_service_owner(best["service_name"])
    steps = suggest_next_steps(best["service_name"])

    ambiguous = False
    if len(retrieved) > 1:
        score_gap = int(retrieved.iloc[0]["retrieval_score"]) - int(retrieved.iloc[1]["retrieval_score"])
        ambiguous = score_gap <= HYPERPARAMS["ambiguity_margin"] or best["jurisdiction_level"] == "Mixed"

    reasoning = owner["reasoning_summary"]
    if ambiguous:
        reasoning = (
            reasoning +
            " The top matches are close together, so the assistant should hedge and may need a clarification question."
        )

    confidence = HYPERPARAMS["default_confidence"]
    if ambiguous:
        confidence = 0.58 if best["jurisdiction_level"] != "Mixed" else 0.5

    return {
        "service_name": owner["service_name"],
        "jurisdiction_level": owner["jurisdiction_level"],
        "responsible_body": owner["responsible_body"],
        "confidence": confidence,
        "reasoning_summary": reasoning,
        "next_steps": steps["next_steps"],
        "sources": owner["sources"],
        "retrieved_candidates": retrieved[["service_name", "jurisdiction_level", "retrieval_score"]].to_dict(orient="records"),
    }

route_question("Who do I contact about garbage pickup?")

{'service_name': 'garbage pickup',
 'jurisdiction_level': 'Region',
 'responsible_body': 'Region of Waterloo Waste Management',
 'confidence': 0.78,
 'reasoning_summary': 'Residential garbage collection schedule, missed pickup, and waste cart information.',
 'next_steps': ['Check the regional waste collection schedule; report a missed pickup if applicable',
  'Verify the latest details on the official source: https://www.regionofwaterloo.ca/'],
 'sources': ['https://www.regionofwaterloo.ca/'],
 'retrieved_candidates': [{'service_name': 'garbage pickup',
   'jurisdiction_level': 'Region',
   'retrieval_score': 2}]}

## Example Questions

The following examples include both clear and ambiguous questions. This makes it easier to see when the workflow can answer with higher confidence and when it should be more careful.

In [5]:
sample_questions = [
    "Who do I contact about garbage pickup?",
    "Is childcare a city or provincial service?",
    "Who handles public transit?",
]

sample_results = pd.DataFrame([
    {
        "question": question,
        **route_question(question),
    }
    for question in sample_questions
])

sample_results[[
    "question",
    "service_name",
    "jurisdiction_level",
    "responsible_body",
    "confidence",
]]

,question,service_name,jurisdiction_level,responsible_body,confidence
0,Who do I contact about garbage pickup?,garbage pickup,Region,Region of Waterloo Waste Management,0.78
1,Is childcare a city or provincial service?,child care subsidies,Province,Government of Ontario,0.58
2,Who handles public transit?,public transit,Mixed,Grand River Transit / Regional municipality co...,0.50


## Gemini Agent Layer

This notebook uses Gemini as the model layer for the agent workflow. The API key is loaded from `.env` first, and it can also come from the shell environment if it is already configured there.

In [6]:
from google import genai
from google.genai import types

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY is required for this notebook.")

client = genai.Client(api_key=api_key)

tool_declarations = [
    types.Tool(function_declarations=[{
        "name": "search_service_index",
        "description": "Search the local civic service catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"],
        },
    }]),
    types.Tool(function_declarations=[{
        "name": "lookup_service_owner",
        "description": "Look up the owner of a civic service.",
        "parameters": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string"}
            },
            "required": ["service_name"],
        },
    }]),
    types.Tool(function_declarations=[{
        "name": "suggest_next_steps",
        "description": "Suggest next steps once the service is known.",
        "parameters": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string"}
            },
            "required": ["service_name"],
        },
    }]),
]

print("Gemini client and tool declarations are ready.")

Gemini client and tool declarations are ready.


## Evaluation And Artifact Export

To check whether the workflow is working reasonably well, we run it on the benchmark set and calculate a few simple metrics. The outputs are saved to the `artifacts/` directory so they can be reused later.

In [7]:
predictions = []
for _, row in eval_df.iterrows():
    pred = route_question(row["question"])
    predictions.append({
        "question": row["question"],
        "expected_service_name": row["expected_service_name"],
        "expected_jurisdiction_level": row["expected_jurisdiction_level"],
        "expected_responsible_body": row["expected_responsible_body"],
        "predicted_service_name": pred["service_name"],
        "predicted_jurisdiction_level": pred["jurisdiction_level"],
        "predicted_responsible_body": pred["responsible_body"],
        "confidence": pred["confidence"],
        "sources": pred["sources"],
    })

pred_df = pd.DataFrame(predictions)
metrics = {
    "jurisdiction_accuracy": float(
        (pred_df["predicted_jurisdiction_level"] == pred_df["expected_jurisdiction_level"]).mean()
    ),
    "responsible_body_accuracy": float(
        (pred_df["predicted_responsible_body"] == pred_df["expected_responsible_body"]).mean()
    ),
    "source_presence_rate": float(pred_df["sources"].apply(lambda value: len(value) > 0).mean()),
    "n_examples": int(len(pred_df)),
}

(ARTIFACTS_DIR / "service_catalog.cleaned.json").write_text(
    clean_catalog.to_json(orient="records", indent=2)
)
(ARTIFACTS_DIR / "eval_predictions.json").write_text(
    pred_df.to_json(orient="records", indent=2)
)
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

metrics

{'jurisdiction_accuracy': 0.875,
 'responsible_body_accuracy': 0.875,
 'source_presence_rate': 1.0,
 'n_examples': 8}

## DVC Orchestration

In this project, DVC is used to manage the workflow as a small reproducible pipeline instead of running each script manually without tracking. The main pipeline definition is written in `dvc.yaml`, where the work is separated into three stages: `prepare_data`, `run_agent_eval`, and `report_metrics`.

The `prepare_data` stage reads the original service catalog and writes a cleaned artifact called `service_catalog.cleaned.json`. This stage is important because the later retrieval and evaluation steps depend on the cleaned version of the data rather than the raw CSV alone.

The `run_agent_eval` stage uses the cleaned catalog together with the evaluation set to produce `eval_predictions.json` and `metrics.json`. In this stage, DVC also watches the parameters defined in `params.yaml`, such as `retrieval.top_k`, `retrieval.min_score`, `retrieval.ambiguity_margin`, `agent.use_tool_calling`, and `evaluation.max_examples`. This means that if we change one of these values, DVC can detect that the experiment settings changed and rerun the affected stage.

The `report_metrics` stage reads `metrics.json` and generates a short markdown report. This makes the final evaluation easier to review because the key numbers are saved in a readable format instead of only appearing in terminal output.

A useful part of DVC in this assignment is that it connects code, data, parameters, and outputs in one place. Instead of saying that we changed the retrieval depth or confidence setting, we can store the values in `params.yaml`, rerun the pipeline, and compare the new artifacts and metrics in a more organized way.

In practice, the workflow can be managed with commands such as `dvc repro` to rerun the pipeline, `dvc metrics show` to inspect the saved metrics, and `dvc params diff` or Git history to compare experimental settings across versions. This is helpful for agent-based systems because even small changes in retrieval or routing logic can affect the final results, and DVC gives a clearer record of those changes.

In [8]:
print((ROOT / "dvc.yaml").read_text())
print((ROOT / "params.yaml").read_text())

stages:
  prepare_data:
    cmd: python scripts/prepare_data.py
    deps:
      - scripts/prepare_data.py
      - data/service_catalog.csv
    outs:
      - artifacts/service_catalog.cleaned.json

  run_agent_eval:
    cmd: python scripts/run_agent_eval.py
    deps:
      - scripts/run_agent_eval.py
      - eval/service_eval_set.csv
      - artifacts/service_catalog.cleaned.json
    params:
      - retrieval.top_k
      - retrieval.min_score
      - retrieval.keyword_weight
      - retrieval.ambiguity_margin
      - agent.model_name
      - agent.use_tool_calling
      - agent.require_sources
      - agent.default_confidence
      - agent.unclear_confidence
      - evaluation.max_examples
    outs:
      - artifacts/eval_predictions.json
    metrics:
      - artifacts/metrics.json

  report_metrics:
    cmd: python scripts/report_metrics.py
    deps:
      - scripts/report_metrics.py
      - artifacts/metrics.json
    outs:
      - artifacts/metrics_report.md

retrieval:
  top_k: 3
  m

## Talking Points

**1. What is the smallest useful definition of an AI agent?**  
In my understanding, an AI agent is more than a chatbot that only gives text. A useful agent should be able to decide what action to take, for example retrieving data, calling a tool, and then organizing the final answer.

**2. When should you prefer retrieval over memory?**  
Retrieval is better when the answer depends on specific facts, official records, or information that may change over time. In this civic case, it is safer to use the local service catalog than to rely only on model memory.

**3. When is a function call better than asking the model to answer directly?**  
A function call is better when we already have a clear source of truth or a fixed step we want the system to follow. In this notebook, ownership lookup and next-step suggestions are more reliable when they come from tool functions connected to the dataset.

**4. What does a grounded answer mean in a civic context?**  
A grounded answer means the response is supported by the service data and official source links, not by guessing. For public-service questions, this is important because a wrong answer can send people to the wrong government office.

**5. Why might a single-agent system outperform a multi-agent system on a student project?**  
For a student project, a single-agent workflow is usually easier to build, test, and debug. A multi-agent system may look more advanced, but it can also add extra complexity without improving the final result very much.

**6. What parts of this architecture should become DVC pipeline stages later?**  
The data preparation step, evaluation step, and report generation step should all become DVC stages. These parts produce files and metrics, so it makes sense to track them in a reproducible pipeline.

**7. Which parts of this problem are classification problems?**  
Identifying the jurisdiction level is a classification task, and predicting the responsible body is also similar to classification. Another classification-like part is deciding whether the case is clear, mixed, or unclear.

**8. Which parts require retrieval?**  
Retrieval is needed when the system tries to match the question to the correct service and find the related supporting information. It is also necessary for attaching the correct source link to the answer.

**9. Which parts benefit from tool use?**  
Tool use helps in the parts that should be consistent and testable, such as searching the catalog, checking service ownership, and preparing the next steps. This reduces the chance that the model will make up details.

**10. Which outputs should be deterministic and machine-readable?**  
The main output fields should be machine-readable, such as service name, jurisdiction level, responsible body, confidence, sources, and next steps. This makes the workflow easier to evaluate and compare.

## Conclusion

Overall, this notebook shows how a small civic-service dataset can be used to build a simple agent workflow. The final system is not very complex, but it is grounded, structured, and easier to evaluate than a prompt-only approach.